# In this notebook, I will create a dataset of the remaining yellowballs in the MIRION catalog and my prediction for their physical properties

In [213]:
import pandas as pd
from ratio_function import RatioGenerator, LogRatioGenerator
import matplotlib.pyplot as plt
import itertools
import numpy as np
import argparse
from pyhere import here
import pickle
from sklearn.model_selection import train_test_split
from sklearn import set_config
from skopt import BayesSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from skopt.space import Categorical, Real, Integer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from skopt.plots import plot_convergence, plot_objective, plot_evaluations
from sklearn.metrics import mean_pinball_loss, make_scorer
from sklearn.model_selection import cross_val_score
import optuna

The results dictionary has 4 keys: best_quantile_error, best_quantile_params, best_mean_error, best_mean_params

In [214]:
phot_full = pd.read_csv(here("data/cleaned", "MIRION_cleaned_everything.csv"))
properties = phot_full.copy()

flux_cols = ['F8', 'F12', 'F24', 'F70']

X = phot_full[['F8','e_F8','F12','e_F12','F24','e_F24','F70','e_F70','N8', 'N12', 'N24', "N70", "DIST", 'f_MULTI','f_CEXT']]

for wavelength in ['8', '12', '24', '70']:
        X[f'se_F{wavelength}'] = X[f'e_F{wavelength}'] / np.sqrt(X[f'N{wavelength}'])

In [215]:
print(len(X))

3944


In [216]:
results_dict = {}
for response in ["LRATIO", "LM", "L_BOL", "MASS", "DIAM", "SURF_DENS", "TEMP", "T_BOL"]:
    with open(here("pipeline/scripts/optuna/final_preds_and_models/results", f"{response}_final_predictive_models.pkl"), "rb") as file:
        results_file = pickle.load(file)

        results_dict[response] = results_file

### Now that I have all the model information that I need, I will organize the creation of a dataframe consisting of the predictions for the yellowballs that don't have physical properties.

To do this, I first extract all the feature information that I need to predict the yellowballs. I clean the data, and then filter it to only contain that ones that I don't have longer wavelength information on. Then, I train a model on the data I have and predict on the data I don't. Using a for loop, I can do this all in one go.

In [217]:
phot_colspecs = [
    (0, 4),    # YB
    (5, 14),   # GLON
    (15, 23),  # GLAT
    (24, 31),  # MWPR
    (60, 68),  # F8
    (69, 77),  # e_F8
    (78, 85),  # F12
    (86, 94),  # e_F12
    (95, 102), # F24
    (103, 110),# e_F24
    (111, 120),# F70
    (121, 130),# e_F70
    (131, 133),# N8
    (134, 136),# N12
    (137, 139),# N24
    (140, 141),# N70
    (142, 146),# f_SAT
    (147, 148),# f_MULTI
    (149, 153),# f_NOSRC
    (154, 158),# f_PCONF
    (159, 160),# f_CEXT
]
phot_colnames = [
    "YB", "GLON", "GLAT", "MWPR",
    "F8", "e_F8", "F12", "e_F12", "F24", "e_F24", "F70", "e_F70",
    "N8", "N12", "N24", "N70",
    "f_SAT", "f_MULTI", "f_NOSRC", "f_PCONF", "f_CEXT"
]
phot = pd.read_fwf(
    here("data/uncleaned", "MRT-phot.txt"),
    colspecs=phot_colspecs,
    names=phot_colnames,
    skiprows=38,
    encoding='utf-8-sig',
    dtype={'f_SAT': str, 'f_PCONF': str, 'f_NOSRC': str},
    na_values=[999.0000]
)

phot['f_SAT_F8'] = phot['f_SAT'].str[0].astype(int)
phot['f_SAT_F12'] = phot['f_SAT'].str[1].astype(int)
phot['f_SAT_F24'] = phot['f_SAT'].str[2].astype(int)
phot['f_SAT_F70'] = phot['f_SAT'].str[3].astype(int)
phot['f_NOSRC_F8'] = phot['f_NOSRC'].str[0].astype(int)
phot['f_NOSRC_F12'] = phot['f_NOSRC'].str[1].astype(int)
phot['f_NOSRC_F24'] = phot['f_NOSRC'].str[2].astype(int)
phot['f_NOSRC_F70'] = phot['f_NOSRC'].str[3].astype(int)
phot['f_PCONF_F8'] = phot['f_PCONF'].str[0].astype(int)
phot['f_PCONF_F12'] = phot['f_PCONF'].str[1].astype(int)
phot['f_PCONF_F24'] = phot['f_PCONF'].str[2].astype(int)
phot['f_PCONF_F70'] = phot['f_PCONF'].str[3].astype(int)

phot.loc[phot['f_SAT_F8'] == 1, 'F8'] = np.nan
phot.loc[phot['f_SAT_F12'] == 1, 'F12'] = np.nan
phot.loc[phot['f_SAT_F24'] == 1, 'F24'] = np.nan
phot.loc[phot['f_SAT_F70'] == 1, 'F70'] = np.nan
phot.loc[phot['f_NOSRC_F8'] == 1, 'F8'] = np.nan
phot.loc[phot['f_NOSRC_F12'] == 1, 'F12'] = np.nan
phot.loc[phot['f_NOSRC_F24'] == 1, 'F24'] = np.nan
phot.loc[phot['f_NOSRC_F70'] == 1, 'F70'] = np.nan
phot.loc[phot['f_PCONF_F8'] == 1, 'F8'] = np.nan
phot.loc[phot['f_PCONF_F12'] == 1, 'F12'] = np.nan
phot.loc[phot['f_PCONF_F24'] == 1, 'F24'] = np.nan
phot.loc[phot['f_PCONF_F70'] == 1, 'F70'] = np.nan

phot.loc[phot['f_SAT_F8'] == 1, 'e_F8'] = np.nan
phot.loc[phot['f_SAT_F12'] == 1, 'e_F12'] = np.nan
phot.loc[phot['f_SAT_F24'] == 1, 'e_F24'] = np.nan
phot.loc[phot['f_SAT_F70'] == 1, 'e_F70'] = np.nan
phot.loc[phot['f_NOSRC_F8'] == 1, 'e_F8'] = np.nan
phot.loc[phot['f_NOSRC_F12'] == 1, 'e_F12'] = np.nan
phot.loc[phot['f_NOSRC_F24'] == 1, 'e_F24'] = np.nan
phot.loc[phot['f_NOSRC_F70'] == 1, 'e_F70'] = np.nan
phot.loc[phot['f_PCONF_F8'] == 1, 'e_F8'] = np.nan
phot.loc[phot['f_PCONF_F12'] == 1, 'e_F12'] = np.nan
phot.loc[phot['f_PCONF_F24'] == 1, 'e_F24'] = np.nan
phot.loc[phot['f_PCONF_F70'] == 1, 'e_F70'] = np.nan

phot = phot.drop(columns=[
    'f_SAT_F8',
    'f_SAT_F12',
    'f_SAT_F24',
    'f_SAT_F70',
    'f_NOSRC_F8',
    'f_NOSRC_F12',
    'f_NOSRC_F24',
    'f_NOSRC_F70',
    'f_PCONF_F8',
    'f_PCONF_F12',
    'f_PCONF_F24',
    'f_PCONF_F70',
    'GLON',
    'GLAT',
    'MWPR',
    'f_SAT',
    'f_NOSRC',
    'f_PCONF'
    ]
)

print(len(phot))

dist_colspecs = [
    (0, 4),    #ID 
   (5, 10),    #DIST 
  (11, 16),    #e_DIST      
  (17, 22),    #DIST_C	 
  (23, 29),    #DIST_M 
  (30, 36),    #e_DIST_M  
  (37, 48),    #STAT_M   
  (49, 53),    #PFAR 
  (54, 59),    #DIST_R1   
  (60, 65),    #e_DIST_R1
  (66, 70),    #PINT_R1
  (71, 76),    #ARM_R1
  (77, 82),    #DIST_R2
  (83, 87),    #e_DIST_R2   
  (88, 92),    #PINT_R2   
  (93, 98)     #ARM_R2
]
dist_colnames = ['YB', 'DIST', 'e_DIST', 'DIST_C', 'DIST_M', 'e_DIST_M', 'STAT_M', 'PFAR', 'DIST_R1', 'e_DIST_R1', 'PINT_R1', 'ARM_R1', 'DIST_R2', 'e_DIST_R2', 'PINT_R2', 'ARM_R2']

dist = pd.read_fwf(
    here("data/uncleaned", "MRT-dist.txt"),
    colspecs=dist_colspecs,
    names=dist_colnames,
    skiprows=28,
    encoding='utf-8-sig',
    na_values=[999.0000]
)

dist = dist.drop(columns = ['e_DIST', 'DIST_C', 'DIST_M', 'e_DIST_M', 'STAT_M', 'PFAR', 'DIST_R1', 'e_DIST_R1', 'PINT_R1', 'ARM_R1', 'DIST_R2', 'e_DIST_R2', 'PINT_R2', 'ARM_R2'])

cross_matched = pd.read_csv(here('data/uncleaned', 'MIRION_meta_din_1.csv'))

6176


In [218]:
print(6167-3945)

2222


In [219]:
phot_for_preds = phot[~phot['YB'].isin(cross_matched['YB'])]

for wavelength in ['8', '12', '24', '70']:
    phot_for_preds[f'se_F{wavelength}'] = phot_for_preds[f'e_F{wavelength}'] / np.sqrt(phot_for_preds[f'N{wavelength}'])

phot_for_preds = phot_for_preds.merge(dist, on='YB', how='left')
prediction_ybs = phot_for_preds['YB']
phot_for_preds = phot_for_preds[['F8','e_F8','F12','e_F12','F24','e_F24','F70','e_F70','N8', 'N12', 'N24', "N70", "DIST", 'f_MULTI','f_CEXT', 'se_F8', 'se_F12', 'se_F24', 'se_F70']]

### Now that I have the data, I can do the models

In [220]:
# for some reason, even in the logs for model_fit_3106003 (this model)
# these four are missing even though they show up there in the logs
# I don't know why
results_dict['LRATIO']['best_mean_params'] = {'error_threshold': 4.85, 'scalers': 'standard', 'ratios': 'norm_ratio', 'imputers': 'median', 'n_estimators': 2050, 'learning_rate': 0.003078967214304464, 'max_depth': 4, 'min_child_weight': 10, 'subsample': 0.9, 'colsample_bytree': 0.9, 'colsample_bylevel': 0.8, 'reg_alpha': 5.879802149885816e-05, 'reg_lambda': 0.0033440181467403666, 'gamma': 2.955215600426458e-06}

results_dict['LRATIO']['best_quantile_params'] = {'error_threshold': 2.6, 'scalers': 'standard', 'ratios': 'norm_ratio', 'imputers': 'median', 'n_estimators': 1750, 'learning_rate': 0.005702808748129885, 'max_depth': 3, 'min_child_weight': 6, 'subsample': 0.9, 'colsample_bytree': 0.8, 'colsample_bylevel': 1.0, 'reg_alpha': 0.08544436579847664, 'reg_lambda': 2.419277708822829e-06, 'gamma': 1.5155984928220948e-06}

In [ ]:
alphas = [0.025, 0.5, 0.975]

for response in ["LRATIO", "LM", "L_BOL", "MASS", "DIAM", "SURF_DENS", "TEMP", "T_BOL"]:
    new_df = pd.DataFrame()
    new_df['YB'] = prediction_ybs

    mean_model_params = results_dict[response]['best_mean_params']

    # first, the mean prediction
    # training datat
    if response == "T_BOL":
        properties = properties[properties['T_BOL']<90]
        X_copy = X.copy()
        X_mean = properties.merge(X_copy, on='DIST', how='left')
        y = 
    else:
        X_mean = X.copy()
    
    mean_threshold = mean_model_params['error_threshold']
    for wavelength in ['8', '12', '24', '70']:
        X_mean.loc[X_mean[f'se_F{wavelength}'] > mean_threshold, f'F{wavelength}'] = np.nan


    if response != "TEMP" and response != "T_BOL":
        y =np.log(properties[response])
    elif response == 'TEMP':
        y = properties[response]

    # model
    if mean_model_params['scalers'] == "standard":
        scaler = StandardScaler()
    elif mean_model_params['scalers'] == 'robust':
        scaler = RobustScaler()
    else:
        scaler = 'passthrough'

    if mean_model_params['ratios'] == 'norm_ratio':
        ratio = RatioGenerator(cols=flux_cols)
    else:
        ratio = LogRatioGenerator(cols=flux_cols)

    imputer = None
    if mean_model_params['imputers'] == 'mean':
        imputer == SimpleImputer(strategy='mean')
    elif mean_model_params['imputers'] == 'median': 
        imputer == SimpleImputer(strategy='median')
    else:
        imputer == 'passthrough'

    mean_model = XGBRegressor(random_state=2026)
    valid_mean_params = {k: v for k, v in mean_model_params.items() if k in mean_model.get_params()}
    mean_model.set_params(**valid_mean_params)
    mean_model_pipe = Pipeline([
        ('imputer', imputer),
        ('ratio', ratio),
        ('scaler', scaler),
        ('model', mean_model)
    ])
    mean_model_pipe.fit(X_mean, y)

    mean_preds = mean_model_pipe.predict(phot_for_preds)
    if response == "TEMP" or response == "T_BOL":
        new_df['mean_pred'] = mean_preds
    else:
        new_df['mean_pred'] = np.exp(mean_preds)

    # now the quantile models
    quantile_model_params = results_dict[response]['best_quantile_params']

    X_quant = X.copy()
    quant_threshold = quantile_model_params['error_threshold']
    for wavelength in ['8', '12', '24', '70']:
        X_quant.loc[X_quant[f'se_F{wavelength}'] > quant_threshold, f'F{wavelength}'] = np.nan

    for alpha in alphas:
        if quantile_model_params['scalers'] == "standard":
            scaler = StandardScaler()
        elif quantile_model_params['scalers'] == 'robust':
            scaler = RobustScaler()
        else:
            scaler = 'passthrough'

        if quantile_model_params['ratios'] == 'norm_ratio':
            ratio = RatioGenerator(cols=flux_cols)
        else:
            ratio = LogRatioGenerator(cols=flux_cols)

        imputer = None
        if quantile_model_params['imputers'] == 'mean':
            imputer == SimpleImputer(strategy='mean')
        elif quantile_model_params['imputers'] == 'median': 
            imputer == SimpleImputer(strategy='median')
        else:
            imputer == 'passthrough'

        quantile_model = XGBRegressor(random_state=2026, objective = "reg:quantileerror", quantile_alpha=alpha)
        valid_quant_params = {k: v for k, v in quantile_model_params.items() if k in quantile_model.get_params()}
        quantile_model.set_params(**quantile_model_params)
        quantile_model_pipe = Pipeline([
        ('imputer', imputer),
        ('ratio', ratio),
        ('scaler', scaler),
        ('model', quantile_model)
    ])
        quantile_model_pipe.fit(X_quant, y)

        quant_preds = quantile_model_pipe.predict(phot_for_preds)

        if response == "TEMP" or response == "T_BOL":
            if alpha == 0.5:
                new_df['median_pred'] = quant_preds
            else:
                new_df[f'{alpha}_pred'] = quant_preds
        else:
            if alpha == 0.5:
                new_df['median_pred'] = np.exp(quant_preds)
            else:
                new_df[f'{alpha}_pred'] = np.exp(quant_preds)
    
    new_df.to_csv(here('predictions', f'{response}_final_predictions.csv'), index=False)

c:\Users\alexe\miniforge3\envs\math4025\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:33:56] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "error_threshold", "imputers", "ratios", "scalers" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\alexe\miniforge3\envs\math4025\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:33:59] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "error_threshold", "imputers", "ratios", "scalers" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\alexe\miniforge3\envs\math4025\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:34:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "error_threshold", "imputers", "ratios", "scalers" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\alexe\miniforge3\envs\math4025\Lib\site-packages\xgboost\training.py:200: UserWarn

XGBoostError: [15:35:10] C:\actions-runner\_work\xgboost\xgboost\src\data\data.cc:522: Check failed: p_info->Size() % n_samples == 0 (3909 vs. 0) : Invalid size for `label`:(3909,1). n_samples:31612